Dans ce notebook on va séparer en train et test nos données avant analyse préalable de la base pour éviter toute forme de data leakage. 

In [1]:
import os 
import pandas as pd 
from sklearn.model_selection import train_test_split
import numpy as np
import plotly.graph_objects as go

In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks


'c:\\Users\\leoco\\Documents\\cours\\M2_MOSEF\\ML_theory\\land_value_prediction'

In [3]:
from src.land_value_prediction.train_test_split.analysis_functions import comparer_train_test, identifier_differences_significatives

In [4]:
idf_vf_full = pd.read_parquet("data/processed/idf_vf_full.parquet")
idf_vf_full.head()

,date_mutation,nature_mutation,valeur_fonciere,adresse_numero,adresse_nom_voie,adresse_code_voie,code_postal,code_commune,nom_commune,code_departement,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2021-01-01,Vente,169500.0,28.0,ALL HOCHE,4440,92130.0,92040,Issy-les-Moulineaux,92,...,0.090161,0.075901,0.254014,0.700824,0.835437,0.704796,0.747019,0.376209,1.045371,16106.117647
1,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,0.085686,0.071492,0.182996,0.567644,0.364296,0.331115,0.421018,0.561570,0.784827,6400.116144
2,2021-01-02,Vente,415000.0,109.0,RUE DES COURLIS,1570,95100.0,95018,Argenteuil,95,...,0.085686,0.071492,0.182996,0.567644,0.364296,0.331115,0.421018,0.561570,0.784827,6400.116144
3,2021-01-02,Vente,512000.0,97.0,RUE DE CHARONNE,1888,75011.0,75111,Paris 11e Arrondissement,75,...,0.067020,0.131609,0.145909,0.680558,0.851424,0.607516,0.713166,0.996164,-1.129705,39957.220708
4,2021-01-02,Vente,512000.0,97.0,RUE DE CHARONNE,1888,75011.0,75111,Paris 11e Arrondissement,75,...,0.067020,0.131609,0.145909,0.680558,0.851424,0.607516,0.713166,0.996164,-1.129705,39957.220708


On va créer 3 échantillons :  
- échantillons train et test sur 2021-2024  
- échantillon temporel out_of_sample (2025-S1)  

Séparation train-test et échantillon out of sample :

In [5]:
train_test = idf_vf_full[idf_vf_full['annee'] < 2025].copy()
out_of_sample_df = idf_vf_full[idf_vf_full['annee'] >= 2025].copy()

print(f"Taille du train-test: {len(train_test)} ({len(train_test)/len(idf_vf_full)*100:.1f}%)")
print(f"Taille du out of sample: {len(out_of_sample_df)} ({len(out_of_sample_df)/len(idf_vf_full)*100:.1f}%)")
print(f"\nPériode train-test: {train_test['date_mutation'].min()} à {train_test['date_mutation'].max()}")
print(f"Période out of sample: {out_of_sample_df['date_mutation'].min()} à {out_of_sample_df['date_mutation'].max()}")

print("\nStatistiques prix_m2 - Train-test:")
train_test['prix_m2'].describe().round(2)

Taille du train-test: 622561 (92.5%)
Taille du out of sample: 50188 (7.5%)

Période train-test: 2021-01-01 00:00:00 à 2024-12-31 00:00:00
Période out of sample: 2025-01-02 00:00:00 à 2025-06-30 00:00:00

Statistiques prix_m2 - Train-test:


count    622561.00
mean       6016.69
std        3660.23
min         500.24
25%        3261.63
50%        4772.73
75%        8238.64
max       19998.89
Name: prix_m2, dtype: float64

In [6]:
print("\nStatistiques prix_m2 - Out of sample:")
out_of_sample_df['prix_m2'].describe().round(2)


Statistiques prix_m2 - Out of sample:


count    50188.00
mean      5825.00
std       3543.62
min        504.00
25%       3181.82
50%       4670.33
75%       7914.29
max      19988.70
Name: prix_m2, dtype: float64

Séparer train et test :

In [7]:
# Créer des déciles sur la variable cible pour stratifier le split
idf_vf_full['prix_m2_bins'] = pd.qcut(idf_vf_full['prix_m2'], q=10, labels=False, duplicates='drop')

# Split train/test avec stratification
train, test = train_test_split(
    idf_vf_full, 
    test_size=0.2, 
    shuffle=True,
    random_state=42, 
    stratify=idf_vf_full['prix_m2_bins']
)

# Supprimer la colonne temporaire de bins
train = train.drop('prix_m2_bins', axis=1)
test = test.drop('prix_m2_bins', axis=1)

print(f"Taille du train set: {len(train)} ({len(train)/len(idf_vf_full)*100:.1f}%)")
print(f"Taille du test set: {len(test)} ({len(test)/len(idf_vf_full)*100:.1f}%)")

print("\nStatistiques prix_m2 - Train:")
train['prix_m2'].describe()

Taille du train set: 538199 (80.0%)
Taille du test set: 134550 (20.0%)

Statistiques prix_m2 - Train:


count    538199.000000
mean       6003.199668
std        3653.826395
min         500.240000
25%        3255.785765
50%        4764.705882
75%        8214.285714
max       19998.888889
Name: prix_m2, dtype: float64

In [8]:
print("\nStatistiques prix_m2 - Test:")
test['prix_m2'].describe()


Statistiques prix_m2 - Test:


count    134550.000000
mean       5999.136593
std        3644.720456
min         501.672241
25%        3255.813953
50%        4764.705882
75%        8209.215550
max       19994.074074
Name: prix_m2, dtype: float64

In [9]:
comparison = comparer_train_test(train, test)
comparison

,type,train_mean,test_mean,diff_mean,train_median,test_median,diff_median,train_std,test_std,diff_std,train_min,test_min,train_max,test_max,diff_mean_pct,diff_median_pct,diff_std_pct
variable,,,,,,,,,,,,,,,,,
date_mutation,categorical,<NA>,<NA>,<NA>,2025-03-31 00:00:00,2025-03-31 00:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nature_mutation,categorical,<NA>,<NA>,<NA>,Vente,Vente,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
valeur_fonciere,numeric,393987.832195,395612.682819,-1624.850624,300000.0,300000.0,0.0,352455.681605,353618.438065,-1162.75646,5200.0,6300.0,10500000.0,9639800.0,-0.410718,0.0,-0.328817
adresse_numero,numeric,131.320988,130.284207,1.036781,19.0,19.0,0.0,864.910964,862.16271,2.748254,1.0,1.0,9999.0,9248.0,0.795784,0.0,0.318763
adresse_nom_voie,categorical,<NA>,<NA>,<NA>,RUE DE PARIS,RUE DE PARIS,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
adresse_code_voie,categorical,<NA>,<NA>,<NA>,0040,0040,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
code_postal,numeric,85524.218438,85517.361477,6.856961,91220.0,91220.0,0.0,8396.943389,8398.569749,-1.62636,75001.0,75001.0,95880.0,95880.0,0.008018,0.0,-0.019365
code_commune,categorical,<NA>,<NA>,<NA>,75115,75115,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nom_commune,categorical,<NA>,<NA>,<NA>,Paris 15e Arrondissement,Paris 15e Arrondissement,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [10]:
warnings = identifier_differences_significatives(comparison, threshold_mean=0.1, threshold_std=0.2)

# Afficher les alertes
if warnings['mean']:
    print("⚠️ Variables avec différences de moyenne > 10%:")
    for w in warnings['mean']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['std']:
    print("\n⚠️ Variables avec différences d'écart-type > 20%:")
    for w in warnings['std']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['distribution']:
    print("\n⚠️ Variables avec plage de valeurs différentes:")
    for w in warnings['distribution']:
        print(f"  - {w['variable']}: train={w['train_range']}, test={w['test_range']}")


⚠️ Variables avec différences d'écart-type > 20%:
  - surface_terrain: -22.7%

⚠️ Variables avec plage de valeurs différentes:
  - latitude: train=[48.125164, 49.234761], test=[48.125164, 49.236552]
  - ecart_prix_median_pct: train=[-96.33846153846154, 998.0108499095842], test=[-96.18037135278514, 1048.6486486486488]


In [11]:
fig = go.Figure()

# Ajouter les distributions
fig.add_trace(go.Box(
    y=train['prix_m2'],
    name='Train',
    boxmean='sd',
    marker_color='lightblue'
))

fig.add_trace(go.Box(
    y=test['prix_m2'],
    name='Test',
    boxmean='sd',
    marker_color='lightgreen'
))

fig.add_trace(go.Box(
    y=out_of_sample_df['prix_m2'],
    name='Out of Sample (2025)',
    boxmean='sd',
    marker_color='lightcoral'
))

# Mise en forme
fig.update_layout(
    title='Comparaison de la distribution de prix_m2 entre les échantillons',
    yaxis_title='Prix au m² (€)',
    showlegend=True,
    height=600,
    template='plotly_white'
)

fig.show()

In [12]:
train.to_parquet("data/processed/train_test_out_sample_split/idf_vf_train.parquet", index=False)
test.to_parquet("data/processed/train_test_out_sample_split/idf_vf_test.parquet", index=False)
out_of_sample_df.to_parquet("data/processed/train_test_out_sample_split/idf_vf_out_of_sample.parquet", index=False)